# 03 – Model Training

In this notebook we:

- Use the feature engineering pipeline to prepare training and validation data.
- Train an XGBoost model.
- Evaluate performance using ROC-AUC.
- Inspect feature importance.


In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

from src.feature_engineering import prepare_training_data

plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
X_train, y_train, X_valid, y_valid = prepare_training_data(
    processed_dir="../data/processed",
    models_dir="../models",
)

X_train.shape, X_valid.shape

In [ ]:
model = XGBClassifier(
    n_estimators=400,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=50,
)

In [ ]:
valid_pred_proba = model.predict_proba(X_valid)[:, 1]
roc_auc = roc_auc_score(y_valid, valid_pred_proba)
print(f"Validation ROC-AUC: {roc_auc:.4f}")

In [ ]:
# Plot top 20 features by importance
importance = model.feature_importances_
feature_names = X_train.columns

indices = np.argsort(importance)[::-1][:20]

plt.figure(figsize=(10, 6))
plt.barh(range(len(indices)), importance[indices][::-1])
plt.yticks(range(len(indices)), feature_names[indices][::-1])
plt.xlabel("Importance")
plt.title("Top 20 feature importances")
plt.tight_layout()
plt.show()

In [ ]:
os.makedirs("../models", exist_ok=True)
model_path = "../models/xgb_model.json"
model.save_model(model_path)
print("Model saved to", model_path)

## Interpretation

- ROC-AUC tells us how well the model ranks fraudulent vs non-fraudulent transactions.
- A value close to 1.0 means very good separation.
- Feature importance highlights which features the model finds most useful.

In practice we would also check other metrics like precision and recall,
but ROC-AUC is a strong first indicator for fraud detection.
